# GA4 Silver Layer:



### Purpose: 
Transform bronze GA4 events into clean, analytics-ready silver tables with flattened nested structures.
### What it dose: 
- It's a bronze → silver ETL transformation for GA4 event data: 
- explode the nested param array → 
- filter to relevant keys → 
- pivot key/value pairs into named columns while flattening struct fields → 
- merge (upsert) the result into the silver table, keyed on row hash + timestamp + event name to avoid duplicates.
### Flow: 
Processing Flow:

- Read data from bronze layer
- Flatten nested structures (same transformations as external)
- Drop unnecessary columns
- Filter for Virginia internal traffic and deduplicate
- Merge into delta table

Bronze events → 
Flatten nested structure(event_params, geo, device, traffic_source)→ 
Filter by traffic type →
Deduplicate → 
Write to silver delta tables


### Why Merge Tables?

- **Rerun immunity:** safe to rerun without reprocessing old data
- Incremental processing: Only new bronze data since last run is processed\

In [0]:
%run ./parms_config

In [0]:
create_table = f"""
create table if not exists {catalog}.{silver_schema}.{silver_table} (
    user_pseudo_id string,
    _databricks_row_hash string,
    event_date DATE,
    event_name string,
    event_timestamp timestamp,
    user_first_touch_timestamp timestamp,
    city string,
    continent string,
    country string,
    region string,
    metro string,
    name string,
    medium string,
    source string,
    platform string,
    is_active_user string,
    language string,
    category string,
    mobile_brand_name string,
    operating_system string,
    page_title string,
    page_location string,
    engagement_time_msec string,
    session_id string,
    traffic_type string,
    link_domain string,
    processing_at  timestamp  
  ) """


spark.sql(
    create_table.format(
        catalog=catalog, silver_schema=silver_schema, silver_table=silver_table
    )
)

In [0]:
def read_bronze(catalog, bronze_schema, bronze_table):
    # Reads raw bronze events, pivots selected event_params into columns, one row per event.

    query = f"""select
            user_pseudo_id,
             _databricks_row_hash,
             CAST(event_date1 AS DATE) as event_date,
             event_name,
             timestamp_micros(event_timestamp) as event_timestamp,
             timestamp_micros(user_first_touch_timestamp) as user_first_touch_timestamp,
             geo.city,
             geo.continent,
             geo.country,
             geo.region,
             geo.metro,
             traffic_source.name ,
             traffic_source.medium,
             traffic_source.source,
             platform,
             is_active_user,
             device.language as language,
             device.category,
             device.mobile_brand_name,
             device.operating_system,
             MAX(CASE WHEN key = 'page_title' THEN param_value END) AS page_title,
             MAX(CASE WHEN key = 'page_location' THEN param_value END) AS page_location,
             MAX(CASE WHEN key = 'engagement_time_msec' THEN param_value END) AS engagement_time_msec,
             MAX(CASE WHEN key = 'session_id' THEN param_value END) AS session_id,
             MAX(CASE WHEN key = 'traffic_type' THEN param_value END) AS traffic_type,
             Max(CASE WHEN key = 'link_domain' THEN param_value END) AS link_domain,
             current_timestamp() as processing_at
             

         from
         (
         select *, param.key, to_date(event_date, 'yyyyMMdd') as event_date1,
            (CASE WHEN param.value.string_value IS NOT NULL THEN param.value.string_value
                  WHEN param.value.int_value IS NOT NULL THEN CAST(param.value.int_value AS STRING)
                  WHEN param.value.float_value IS NOT NULL THEN CAST(param.value.float_value AS STRING)
                  WHEN param.value.double_value IS NOT NULL THEN CAST(param.value.double_value AS STRING)
                  ELSE NULL END) as param_value

         from
         (select *, explode(event_params) as param from workspace.bronze.events) all_events

         where param.key in ('page_title','page_location','engagement_time_msec','session_id','traffic_type','link_domain')

         ) rng
         group by
            user_pseudo_id,
             _databricks_row_hash,
             CAST(event_date1 AS DATE) ,
             event_name,
             timestamp_micros(event_timestamp) ,
             timestamp_micros(user_first_touch_timestamp),
             geo.city,
             geo.continent,
             geo.country,
             geo.region,
             geo.metro,
             traffic_source.name ,
             traffic_source.medium,
             traffic_source.source,
             platform,
             is_active_user,
             device.language ,
             device.category,
             device.mobile_brand_name,
             device.operating_system """

    raw = spark.sql(query)
    return raw

 This cell does three things:
   1. Reads bronze-layer GA4 events via read_bronze() and removes exact duplicate rows with dropDuplicates().
   2. Splits rows into two groups:
      - df_ready: rows where BOTH event_date and event_name are non-null (clean, analytics-ready data).
       Registered as a temp view "df_ready" so the downstream MERGE (Cell 7) can reference it.
      - df_bad: rows where event_date OR event_name is null (quarantined bad data for later inspection).

In [0]:

raw = read_bronze(catalog,bronze_schema,bronze_table)
df_ready = raw.filter(raw.event_date.isNotNull() & raw.event_name.isNotNull())
df_ready.createOrReplaceTempView("df_ready")

df_bad = raw.filter(raw.event_date.isNull() | raw.event_name.isNull())



In [0]:
query = f"""
    merge into {catalog}.{silver_schema}.{silver_table} tgt
      using (
         select* from df_ready
        ) src
      on tgt._databricks_row_hash = src._databricks_row_hash 
      and tgt.event_timestamp = src.event_timestamp 
      and tgt.event_name=src.event_name
      when matched then update set *
      when not matched then insert *

  """
spark.sql(query)

### QA

In [0]:
silver_count = spark.sql(f"select count(*) from {catalog}.{silver_schema}.{silver_table}").collect()[0][0]
bronze_count = spark.sql(f"select count(*) from {catalog}.{bronze_schema}.{bronze_table}").collect()[0][0]
assert silver_count == bronze_count


